In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) Plan zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich noch ändern.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>
## Überblick
In der Quantenchemie befasst sich das Problem der Elektronenstruktur mit der Lösung der elektronischen Schrödinger-Gleichung – also der Quantenwellenfunktionen, die das Verhalten der Elektronen im System beschreiben. Diese Wellenfunktionen sind Vektoren komplexer Amplituden, wobei jede Amplitude dem Beitrag einer möglichen Elektronenkonfiguration entspricht.

Der Grundzustand ist die Wellenfunktion niedrigster Energie des Systems und hat bei der Untersuchung molekularer Systeme eine besondere Bedeutung. Der genaueste Ansatz zur Berechnung des Grundzustands berücksichtigt alle möglichen Elektronenkonfigurationen, wird aber für größere Systeme schnell unlösbar, da die Anzahl der Konfigurationen exponentiell mit der Systemgröße wächst.

Der Handover Iterative Variational Quantum Eigensolver (HI-VQE) ist eine innovative hybride Quanten-klassische Methode zur genauen Schätzung des Grundzustands molekularer Systeme. Er verbindet Quantenhardware mit klassischem Computing: Quantenprozessoren erkunden effizient Kandidaten-Elektronenkonfigurationen, während die resultierende Wellenfunktion auf klassischen Rechnern berechnet wird. Durch die Erzeugung kompakter, aber chemisch genauer Wellenfunktionen fördert HI-VQE Forschung und Entdeckungen in der Quantenchemie und Materialwissenschaft.

![Bild mit einem Überblick über den HI-VQE-Algorithmus von Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE reduziert die Rechenkomplexität des Elektronenstrukturproblems, indem es den Grundzustand effizient und mit hoher Genauigkeit schätzt. Der Fokus liegt auf einer sorgfältig ausgewählten Teilmenge der relevantesten Elektronenkonfigurationen, was sowohl Genauigkeit als auch Effizienz optimiert.

Durch die Kombination der Stärken klassischer und Quantenrechner verfeinert HI-VQE iterativ die aktuelle Schätzung der Wellenfunktion. Seine einzigartigen Techniken zur Teilraumkonstruktion machen die Konfigurationsauswahl effizienter, sodass Nutzerinnen und Nutzer mehr Rechenkontrolle und verbesserte Genauigkeit in Quantenchemiesimulationen erhalten.

Wenn du den Algorithmus eingehender kennenlernen möchtest, kannst du [das zugehörige Forschungspapier lesen](https://arxiv.org/abs/2503.06292)
## Beschreibung
Die Anzahl der Elektronenkonfigurationen eines molekularen Systems wächst exponentiell mit der Systemgröße. Bei bestimmten elektronischen Zuständen, wie dem Grundzustand, tragen jedoch häufig nur wenige Konfigurationen wesentlich zur Energie des Zustands bei. Methoden der ausgewählten Konfigurationswechselwirkung (Selected Configuration Interaction, SCI) nutzen diese Spärlichkeit, um Rechenkosten zu senken, indem sie die relevantesten Konfigurationen identifizieren und auf sie fokussieren. Diese Teilmenge der Konfigurationen wird als Teilraum (Subspace) bezeichnet.

HI-VQE nutzt die inhärente Effizienz von Quantencomputern bei der Darstellung molekularer Systeme, um die Teilraumsuche zu unterstützen. Es verbindet klassische und Quanten-Subroutinen, um das Elektronenstrukturproblem mit hoher Genauigkeit zu lösen. Im Gegensatz zu bestehenden Quanten-SCI-Methoden kombiniert HI-VQE Variationstraining, iterative Teilraumkonstruktion und Vordiagonalisierungs-Konfigurations-Screening, um die Effizienz durch Reduzierung von Quantenmessungen, Iterationen und klassischen Diagonalisierungskosten zu steigern. HI-VQE kann daher auf größere molekulare Systeme angewendet werden, die mehr Qubits benötigen, und senkt die Kosten zur Lösung eines Problems gegebener Größe auf denselben Genauigkeitsgrad.

![Bild mit einer detaillierten Beschreibung der einzelnen Schritte des HI-VQE-Algorithmus von Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Um den Grundzustand eines Systems zu berechnen, verwendet HI-VQE zunächst das klassische Chemiepaket PySCF, um aus nutzergegebenen Eingaben – wie der Molekülgeometrie und weiteren molekularen Informationen – eine molekulare Darstellung zu erzeugen. Anschließend tritt es in eine hybride Quanten-klassische Optimierungsschleife ein, die iterativ einen Teilraum verfeinert, um den Grundzustand optimal darzustellen und gleichzeitig die Anzahl der enthaltenen Konfigurationen zu minimieren. Die Schleife läuft bis Konvergenzkriterien – wie Teilraumgröße oder Energiestabilität – erfüllt sind; danach werden die berechnete Grundzustandswellenfunktion und Energie ausgegeben. Diese Ergebnisse können genutzt werden, um genaue Potentialenergieflächen zu konstruieren und weitere Analysen des Systems durchzuführen.

Die Optimierungsschleife konzentriert sich darauf, die Parameter eines Quantum Circuits anzupassen, um einen hochwertigen Teilraum zu erzeugen. HI-VQE bietet drei Quantum Circuit-Optionen: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) und [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Die Optimierung wird nahe dem Hartree-Fock-Referenzzustand initialisiert, da dieser sich allgemein gut eignet. Der Circuit wird dann auf einem Quantengerät ausgeführt und Konfigurationen werden aus dem resultierenden Quantenzustand gesampelt, bevor sie als binäre Zeichenketten zurückgegeben werden. Aufgrund von Quantengeräterauschen können einige gesampelte Konfigurationen physikalisch ungültig sein und die Elektronenzahl oder den Spin nicht erhalten. HI-VQE begegnet diesem Problem mit dem Konfigurationswiederherstellungsprozess aus dem [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview)-Paket, sodass Nutzerinnen und Nutzer ungültige Konfigurationen entweder korrigieren oder verwerfen können.

Die gültigen Konfigurationen durchlaufen dann einen optionalen Screening-Schritt, um jene zu entfernen, die voraussichtlich nur minimal beitragen. Dies reduziert die Dimension des Teilraums und senkt dadurch die Kosten des Diagonalisierungsschritts. Wenn das Screening aktiviert ist, wird ein vorläufiger Teilraum-Hamiltonoperator aus den gültigen Konfigurationen aufgebaut und eine Diagonalisierung mit sehr lockeren Abbruchkriterien durchgeführt. Obwohl die Genauigkeit der resultierenden Amplituden für jede Konfiguration gering ist, eignet sie sich gut dafür vorherzusagen, welche Konfigurationen in dieser Iteration aus dem Teilraum herausgelassen werden sollen – und sie ist schnell zu berechnen.

Die ausgewählten Konfigurationen werden dem Teilraum hinzugefügt und der Hamiltonoperator des Systems in diesen Teilraum projiziert. Der Teilraum wird iterativ aktualisiert und behält dabei die relevantesten Konfigurationen über Iterationen hinweg. Dieser Ansatz unterscheidet sich von alternativen Methoden, da der Quantum Circuit nicht bei jedem Schritt den vollständigen Grundzustand approximieren muss.

Anschließend wird der Teilraum-Hamiltonoperator klassisch diagonalisiert, um den niedrigsten Eigenwert und den zugehörigen Eigenvektor zu erhalten – eine Approximation des Grundzustands und seiner Energie. Da sich die Qualität des Teilraums durch die Iterationen verbessert, nähert sich der berechnete Grundzustand immer mehr dem wahren Grundzustand an. In diesem Schritt kann ein zusätzlicher Screening-Schritt durchgeführt werden, um Konfigurationen aus dem Teilraum zu entfernen, die keinen wesentlichen Beitrag zum berechneten Grundzustand leisten. Dieser Schritt stellt sicher, dass der in die nächste Iteration übergehende Teilraum so kompakt wie möglich ist. Die Bewertung erfolgt anhand der Amplituden aus der Diagonalisierung, da diese den Wichtigkeitsbeitrag jeder Konfiguration zum berechneten Grundzustand darstellen.

Eine Konvergenzprüfung stellt dann fest, ob weiteres Training die Ergebnisse verbessern würde. Falls ja, wird ein optionaler klassischer Erweiterungsschritt durchgeführt, die Parameter des Quantum Circuits werden aktualisiert, um die berechnete Energie weiter zu minimieren, und der Prozess wiederholt sich. Der klassische Erweiterungsschritt erzeugt zusätzliche Konfigurationen für den Teilraum und ergänzt damit die vom Quantengerät gesampelten Konfigurationen. Dabei wird zunächst die Konfiguration mit der größten Amplitude in den Diagonalisierungsergebnissen identifiziert, bevor neue Konfigurationen mit einfachen und doppelten Anregungen aus der identifizierten Konfiguration generiert werden. Die gewünschte Anzahl dieser Konfigurationen wird dann dem Teilraum hinzugefügt.

Sobald festgestellt wird, dass die Iterationen konvergiert sind, gibt HI-VQE den berechneten Grundzustand (in Form der Zustände im Teilraum und ihrer Amplituden in der Grundzustandswellenfunktion), seine Energie und ein Energievarianzmaß zurück, das angibt, ob der berechnete Zustand einen Eigenzustand des Hamiltonoperators des Systems bildet.

Nutzerinnen und Nutzer können den verwendeten Quantum Circuit und die Anzahl der Shots pro Quantum Circuit wählen sowie die Teilraumgröße steuern oder die klassische Generierung zusätzlicher Konfigurationen aktivieren, um die quantengenerierten Konfigurationen zu ergänzen. Auf diese Weise lässt sich das Verhalten von HI-VQE an die jeweiligen Anwendungsfälle anpassen.
## Lizenzierung
Bitte beachte, dass die Nutzung dieser Qiskit Function auf Probleme mit höchstens 20 Qubits beschränkt ist, sofern keine Lizenz erworben wird, die ein höheres Limit gewährt.

Bitte sende eine E-Mail an [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com), wenn du Informationen zum Erwerb einer Lizenz einholen möchtest.
## Erste Schritte
Zuerst [beantrage Zugang zur Function](https://forms.office.com/r/zN3hvMTqJ1).
Authentifiziere dich dann mit deinem [IBM Quantum&reg; API-Schlüssel](http://quantum.cloud.ibm.com/) und – vorausgesetzt, du hast dein Konto bereits [in deiner lokalen Umgebung gespeichert](/guides/functions#install-qiskit-functions-catalog-client) – wähle die Qiskit Function wie folgt aus:

In [2]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Beispiel

Das erste Beispiel zeigt, wie die Grundzustandsenergie eines NH3-Moleküls mit dem HI-VQE-Algorithmus berechnet wird.

#### Molekülgeometrie und Optionen definieren

Die Molekülgeometrie von NH3 wird mit kartesischen Koordinaten angegeben, getrennt durch ";" für jedes Atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Weitere Optionen können für das Molekülsystem im folgenden Dictionary-Format definiert und übergeben werden.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Führe die Function mit den Geometrie- und Optionseingaben aus.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Es ist empfehlenswert, die Job-ID der Function auszugeben, damit sie bei Support-Anfragen angegeben werden kann, falls etwas schiefgeht.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


In diesem Beispiel werden 16 Qubits mit 8 Orbitalen der sto3g-Basis für ein NH3-Molekül verwendet.
Überprüfe den [Status](/guides/functions#check-job-status) deiner Qiskit Function-Arbeitslast oder rufe die [Ergebnisse](/guides/functions#retrieve-results) wie folgt ab:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [9]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


Nach Abschluss des Jobs können die Ergebnisse mit der `result()`-Instanz abgerufen werden.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Um auf die Grundzustandsenergie zuzugreifen, verwende den Schlüssel `"energy"`. Der Schlüssel `"eigenvector"` liefert die CI-Koeffizienten mit entsprechender Bitstring-Notation der Elektronenkonfiguration, die unter `"states"` der Ergebnisse gespeichert ist.

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

## Leistung

Dieser Abschnitt zeigt die nachgewiesenen Benchmark-Berechnungen von HI-VQE mit einem 24-Qubit-Fall für Li2S, einem 40-Qubit-Fall für ein N2-Molekül und einem 44-Qubit-Fall für ein FeP-NO-System.

#### Dissoziations-Potentialenergieflächenkurve für ein Li2S-Molekül mit 24 Qubits

Die PES-Kurve wird mit der FCI-Referenz und dem Startwert aus RHF zusammen mit dem Energiefehler gegenüber der FCI-Referenz gezeigt.

![Bild, das zeigt, dass HI-VQE Lösungen innerhalb der chemischen Genauigkeit einer klassischen Referenz-PES-Kurve für das Li2S-System liefert](../docs/images/guides/qunova-chemistry/Li2S_PES.avif).

Die Berechnungen wurden mit den folgenden Geometrien und Optionen durchgeführt.